In [1]:
import pandas as pd

In [3]:
df = pd.read_csv(r"D:\ML PRoject\home_credit_data\data\training_dataset.csv")
print(df.shape)

(307511, 131)


In [8]:
#Missing value analysis
missing_df = pd.DataFrame({
    'column':df.columns,
    'missing_counts':df.isnull().sum(),
    'missing_percentage':round((df.isnull().sum()/len(df))*100,2)
})

missing_df = missing_df.sort_values(by='missing_percentage',ascending=False)
print(missing_df.head(30))

                                            column  missing_counts  \
COMMONAREA_MEDI                    COMMONAREA_MEDI          214865   
COMMONAREA_MODE                    COMMONAREA_MODE          214865   
COMMONAREA_AVG                      COMMONAREA_AVG          214865   
NONLIVINGAPARTMENTS_AVG    NONLIVINGAPARTMENTS_AVG          213514   
NONLIVINGAPARTMENTS_MEDI  NONLIVINGAPARTMENTS_MEDI          213514   
NONLIVINGAPARTMENTS_MODE  NONLIVINGAPARTMENTS_MODE          213514   
FONDKAPREMONT_MODE              FONDKAPREMONT_MODE          210295   
LIVINGAPARTMENTS_MODE        LIVINGAPARTMENTS_MODE          210199   
LIVINGAPARTMENTS_AVG          LIVINGAPARTMENTS_AVG          210199   
LIVINGAPARTMENTS_MEDI        LIVINGAPARTMENTS_MEDI          210199   
FLOORSMIN_MEDI                      FLOORSMIN_MEDI          208642   
FLOORSMIN_AVG                        FLOORSMIN_AVG          208642   
FLOORSMIN_MODE                      FLOORSMIN_MODE          208642   
YEARS_BUILD_MEDI    

In [11]:
missing_df.to_csv(r"D:\ML PRoject\home_credit_data\missing_value_reports.csv", index=False)
print("dataset loaded successfully")

dataset loaded successfully


In [14]:
high_missing = missing_df[missing_df['missing_percentage'] > 60]
print(high_missing)

                                            column  missing_counts  \
COMMONAREA_MEDI                    COMMONAREA_MEDI          214865   
COMMONAREA_MODE                    COMMONAREA_MODE          214865   
COMMONAREA_AVG                      COMMONAREA_AVG          214865   
NONLIVINGAPARTMENTS_AVG    NONLIVINGAPARTMENTS_AVG          213514   
NONLIVINGAPARTMENTS_MEDI  NONLIVINGAPARTMENTS_MEDI          213514   
NONLIVINGAPARTMENTS_MODE  NONLIVINGAPARTMENTS_MODE          213514   
FONDKAPREMONT_MODE              FONDKAPREMONT_MODE          210295   
LIVINGAPARTMENTS_MODE        LIVINGAPARTMENTS_MODE          210199   
LIVINGAPARTMENTS_AVG          LIVINGAPARTMENTS_AVG          210199   
LIVINGAPARTMENTS_MEDI        LIVINGAPARTMENTS_MEDI          210199   
FLOORSMIN_MEDI                      FLOORSMIN_MEDI          208642   
FLOORSMIN_AVG                        FLOORSMIN_AVG          208642   
FLOORSMIN_MODE                      FLOORSMIN_MODE          208642   
YEARS_BUILD_MEDI    

In [16]:
high_missing.to_csv(r"D:\ML PRoject\home_credit_data\high_missing_values.csv", index=False)
print("dataset loaded successfully")

dataset loaded successfully


In [19]:
numerical_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Numerical:", len(numerical_cols))
print("Categorical:", len(categorical_cols))

Numerical: 115
Categorical: 16


In [20]:
print(df['TARGET'].value_counts(normalize = True)*100)

TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64


In [23]:
df.rename(columns={'refussed':'refused_count'},inplace=True)

In [28]:
missing_percentage= (df.isnull().sum()/len(df))*100
drop_col = missing_percentage[missing_percentage > 65].index.tolist()

df.drop(columns=drop_col,inplace=True)
print(df.shape)

(307511, 114)


In [29]:
df.drop(columns=["SK_ID_CURR"],inplace=True)

In [ ]:
X =df.drop(columns=['TARGET'])
y = df['TARGET']

print(X.shape)
print(y.shape)

(307511, 112)
(307511,)


In [ ]:
#Checking the missing values after dropped the columsn
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_cols = X.select_dtypes(
    include=["object"]
).columns

print("Numerical:", len(numerical_cols))
print("Categorical:", len(categorical_cols))

Numerical: 97
Categorical: 15


In [ ]:
#Numerical missing vlaue imputation
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")
X[numerical_cols] = num_imputer.fit_transform(X[numerical_cols])


In [ ]:
#categorical missing value imputation
cat_imputer = SimpleImputer(strategy = "most_frequent")
X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])

In [ ]:
#Checking missing values after imputation
print(X.isnull().sum().sum())

0


In [41]:
print(len(numerical_cols))
print(len(categorical_cols))

97
15


In [47]:
from sklearn.preprocessing import LabelEncoder

Label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    Label_encoders[col] = le
print("Encodign completed")

Encodign completed


In [48]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 112 entries, NAME_CONTRACT_TYPE to refused_count
dtypes: float64(97), int32(15)
memory usage: 245.2 MB


In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=42,stratify = y)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(246008, 112)
(246008,)
(61503, 112)
(61503,)


In [63]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter = 2000, random_state=42)
model.fit(X_train, y_train)

c:\Users\user\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

In [64]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [65]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test,y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_prob))


Accuracy: 0.6155959871876169
Recall: 0.5639476334340383
Precision: 0.1153355027392182
F1 Score: 0.1915053689898092
ROC AUC Score: 0.6250039248527232


In [62]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test,y_pred)
print(cm)

[[56536     2]
 [ 4965     0]]


In [81]:
from sklearn.ensemble import RandomForestClassifier

RF = RandomForestClassifier(n_estimators = 100, max_depth= 10, class_weight='balanced', random_state= 42, n_jobs= -1)
RF.fit(X_train, y_train)
y_pred_rf = RF.predict(X_test)
y_prob_rf = RF.predict_proba(X_test)[:-1]

In [80]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
print("Accuracy:", accuracy_score(y_test,y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_rf))

Accuracy: 0.7209729606685853
Recall: 0.6181268882175227
Precision: 0.16739391294862005
F1 Score: 0.26344478303789864
ROC-AUC: 0.6740657434472594


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=11, random_state=42, eval_metric='logloss')

xgb.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [77]:
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

In [78]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
print("Accuracy:", accuracy_score(y_test,y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_xgb))

Accuracy: 0.7387607108596329
Recall: 0.6503524672708962
Precision: 0.18388382687927107
F1 Score: 0.28670366259711433
ROC-AUC: 0.6984384643475355


In [83]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

print(feature_importance.head(20))

                         feature  importance
20                FLAG_EMP_PHONE    0.055047
40                  EXT_SOURCE_3    0.050389
39                  EXT_SOURCE_2    0.048994
11           NAME_EDUCATION_TYPE    0.028967
1                    CODE_GENDER    0.023049
38                  EXT_SOURCE_1    0.019324
78               FLAG_DOCUMENT_3    0.019082
10              NAME_INCOME_TYPE    0.017029
2                   FLAG_OWN_CAR    0.016210
0             NAME_CONTRACT_TYPE    0.014455
75      DEF_60_CNT_SOCIAL_CIRCLE    0.013610
16                 DAYS_EMPLOYED    0.012867
8                AMT_GOODS_PRICE    0.012807
28   REGION_RATING_CLIENT_W_CITY    0.012206
6                     AMT_CREDIT    0.011989
15                    DAYS_BIRTH    0.011977
46                 FLOORSMAX_AVG    0.010995
62                ELEVATORS_MEDI    0.010714
101    AMT_REQ_CREDIT_BUREAU_QRT    0.010695
73      DEF_30_CNT_SOCIAL_CIRCLE    0.010512


In [84]:
feature_importance.to_csv(r"D:\ML PRoject\home_credit_data\feature_importance_xgb.csv", index=False)

In [88]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

In [99]:
import joblib 
joblib.dump(xgb, r"D:\ML PRoject\home_credit_data\models\xgb_model.pkl")
joblib.dump(RF, r"D:\ML PRoject\home_credit_data\models\rf_model.pkl")
joblib.dump(model,r"D:\ML PRoject\home_credit_data\models\lr_model.pkl")
joblib.dump(num_imputer, r"D:\ML PRoject\home_credit_data\models\num_imputer.pkl")
joblib.dump(cat_imputer, r"D:\ML PRoject\home_credit_data\models\cat_imputer.pkl")
joblib.dump(label_encoders, r"D:\ML PRoject\home_credit_data\models\label_encoders.pkl")

['D:\\ML PRoject\\home_credit_data\\models\\label_encoders.pkl']

In [97]:
loaded_model = joblib.load(r"D:\ML PRoject\home_credit_data\models\xgb_model.pkl")
loaded_model1 =joblib.load(r"D:\ML PRoject\home_credit_data\models\rf_model.pkl")
loaded_model2 =joblib.load(r"D:\ML PRoject\home_credit_data\models\lr_model.pkl")

112


In [105]:
joblib.dump(X.columns.tolist(), r"D:\ML PRoject\home_credit_data\models\feature_columns.pkl")

['D:\\ML PRoject\\home_credit_data\\models\\feature_columns.pkl']

In [107]:
cols = joblib.load(r"D:\ML PRoject\home_credit_data\models\feature_columns.pkl")

print(len(cols))
print(cols[:10])

112
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE']


In [2]:
import pandas as pd

df = pd.read_csv(
    r"D:\ML PRoject\home_credit_data\data\training_dataset.csv"
)

# same cleaning steps as training

df.rename(
    columns={
        "refussed": "refused_count"
    },
    inplace=True,
    errors="ignore"
)

# Drop high missing columns
missing_percent = (
    df.isnull().sum() / len(df)
) * 100

drop_cols = missing_percent[
    missing_percent > 65
].index.tolist()

df.drop(
    columns=drop_cols,
    inplace=True
)

# Remove ID and Target

X = df.drop(
    columns=[
        "SK_ID_CURR",
        "TARGET"
    ]
)

In [ ]:
#Recreating the encoding again
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

categorical_cols = X.select_dtypes(
    include=["object"]
).columns

for col in categorical_cols:

    le = LabelEncoder()

    le.fit(
        X[col].astype(str)
    )

    label_encoders[col] = le

In [ ]:
print(label_encoders["NAME_CONTRACT_TYPE"].classes_)
print(label_encoders["CODE_GENDER"].classes_)

['Cash loans' 'Revolving loans']
['F' 'M' 'XNA']


In [ ]:
import joblib
joblib.dump(label_encoders,r"D:\ML PRoject\home_credit_data\models\label_encoders.pkl")
print("Encoders Saved")

Encoders Saved


In [6]:
enc = joblib.load(r"D:\ML PRoject\home_credit_data\models\label_encoders.pkl")

print(enc["NAME_CONTRACT_TYPE"].classes_)

['Cash loans' 'Revolving loans']
